[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_42_Voice_Agent_Pipelines.ipynb)

# Lesson 42 — Track 4 · Voice Agent Pipelines: ASR → LLM → TTS

**Track 4 overview:** Multimodal agents extend the core agent loop (Lessons 1–41) to handle audio, images, and documents. Voice is the most *latency-sensitive* modality — a user talking to an agent expects < 2 s end-to-end, so every design decision is a latency decision.

**Track 4 roadmap:**

| # | Topic | Key skill |
|---|-------|-----------|
| **L42** | **Voice Pipelines ← today** | ASR → Claude → TTS end-to-end |
| L43 | Image Generation as Agent Tools | DALL-E / SD tools, multimodal reasoning |
| L44 | Document AI | PDF extraction, OCR, visual QA |
| L45 | Track 4 Capstone | Multimodal agent combining all three |

**What you'll build today:**
1. ASR with Whisper — convert spoken audio to text
2. Claude as the voice brain — context-aware conversation
3. TTS synthesis — convert text to speech audio
4. `VoiceAgent` — a full turn-based conversational loop
5. Streaming optimization — overlap LLM + TTS for lower perceived latency
6. `LatencyBudget` profiler — measure and visualise each pipeline stage

**Prerequisites:** Lessons 1–41. API key: `ANTHROPIC_API_KEY`. Optional: `OPENAI_API_KEY` for the Whisper API homework.


## Voice Agent Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                    VOICE AGENT LOOP                          │
│                                                              │
│  Microphone / File                                           │
│        │                                                     │
│        ▼                                                     │
│  ┌──────────┐  text   ┌──────────┐  text   ┌────────────┐   │
│  │   ASR    │ ──────► │  LLM     │ ──────► │    TTS     │   │
│  │ Whisper  │         │  Claude  │         │ gTTS /     │   │
│  └──────────┘         └──────────┘         │ ElevenLabs │   │
│                            │               └────────────┘   │
│                       tool calls                │            │
│                       ┌────┴────┐          Speaker /         │
│                       │  Tools  │          Audio file         │
│                       └─────────┘                           │
└──────────────────────────────────────────────────────────────┘
```

**Three latency budgets** (industry targets for conversational AI):

| Stage | Target | Bottleneck |
|-------|--------|------------|
| ASR | < 300 ms | Model size, audio length |
| LLM first token | < 500 ms | Model size, prompt length |
| TTS first chunk | < 200 ms | Synthesis + network |
| **Perceived total** | **< 1.5 s** | Streaming overlap is key |

**Key insight:** Users perceive latency as *time-to-first-sound*. A streaming pipeline that starts playing TTS while the LLM is still generating feels faster than a non-streaming one, even if total wall-clock time is equal.


In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# openai-whisper  — local Whisper model (no API key needed)
# gTTS            — Google Text-to-Speech (free, needs internet)
# pydub           — audio manipulation helpers
!pip install -q anthropic openai-whisper gTTS pydub 2>/dev/null

import os, time, uuid, re, threading, tempfile, statistics
from dataclasses import dataclass, field
from typing import Optional
from pathlib import Path
from IPython.display import Audio, display

# ── API key ───────────────────────────────────────────────────────────────────
os.environ.setdefault("ANTHROPIC_API_KEY", "YOUR_KEY_HERE")

import anthropic
client = anthropic.Anthropic()
print("✓ Dependencies ready")


## Part 1 — ASR: Speech to Text with Whisper

**Whisper** (OpenAI, 2022) is the standard open-source ASR model. It handles 99 languages, noisy audio, accents, and code-switching with no fine-tuning.

**Model variants** — quality vs speed:

| Model | Params | Colab (T4) speed | WER (en) | Use when |
|-------|--------|-----------------|----------|----------|
| `tiny` | 39 M | ~1 s / min audio | ~14 % | CPU demo |
| `base` | 74 M | ~2 s / min audio | ~9 % | Dev / testing |
| `small` | 244 M | ~5 s / min audio | ~6 % | Production (T4) |
| `medium` | 769 M | ~15 s / min audio | ~4 % | High accuracy |
| `large-v3` | 1.5 B | ~45 s / min audio | ~2 % | Offline batch |

For an interactive voice agent, `base` on T4 GPU hits our < 300 ms budget for utterances under 5 s. For production streaming ASR (< 100 ms), see **Deepgram** or **AssemblyAI** APIs.

**Critical Whisper flags:**
- `temperature=0.0` — greedy decoding, deterministic, fewer hallucinations
- `condition_on_previous_text=False` — prevents hallucination loops on long audio
- `fp16=False` — required on CPU; set `True` only with GPU


In [ ]:
# ── ASR with Whisper ─────────────────────────────────────────────────────────
import whisper
from gtts import gTTS

print("Loading Whisper 'base' model (downloads ~150 MB on first run)...")
t0 = time.time()
asr_model = whisper.load_model("base")
print(f"  Loaded in {time.time()-t0:.1f}s")


def transcribe_audio(audio_path: str, language: str = "en") -> dict:
    """
    Transcribe an audio file to text.

    Returns:
        text            — transcribed string
        language        — detected language code
        duration_s      — audio length in seconds
        asr_latency_s   — transcription wall-clock time
        rtf             — real-time factor (< 1 = faster than real-time, good)
    """
    t0 = time.time()
    result = asr_model.transcribe(
        audio_path,
        language=language,
        fp16=False,                       # set True if GPU available
        temperature=0.0,                  # greedy — deterministic, fewer hallucinations
        condition_on_previous_text=False, # prevents hallucination loops
    )
    latency = time.time() - t0

    audio_data = whisper.load_audio(audio_path)
    duration = len(audio_data) / whisper.audio.SAMPLE_RATE

    return {
        "text": result["text"].strip(),
        "language": result.get("language", language),
        "duration_s": round(duration, 2),
        "asr_latency_s": round(latency, 3),
        "rtf": round(latency / max(duration, 0.001), 2),
    }


def make_test_audio(text: str, path: str = "/tmp/test_input.mp3") -> str:
    """Synthesise a test audio clip to feed into Whisper (simulates mic input)."""
    gTTS(text=text, lang="en", slow=False).save(path)
    return path


# ── Smoke test ────────────────────────────────────────────────────────────────
TEST_UTTERANCE = "Tell me about transformer attention mechanisms in simple terms."
audio_path = make_test_audio(TEST_UTTERANCE)
result = transcribe_audio(audio_path)

print(f"Transcription : '{result['text']}'")
print(f"Audio duration: {result['duration_s']} s")
print(f"ASR latency   : {result['asr_latency_s']} s")
rtf_status = "✓ faster than real-time" if result["rtf"] < 1 else "✗ slower — consider tiny model"
print(f"Real-time factor: {result['rtf']}x  ({rtf_status})")


## Part 2 — Claude as the Voice Brain

Prompt engineering changes significantly for voice:

| Text agent | Voice agent |
|-----------|-------------|
| Markdown fine (bold, tables, code blocks) | Must produce *speakable* plain text |
| Long detailed answers OK | Aim for < 30 s of TTS (≈ 75 words) |
| "See the table above" works | Must be self-contained |
| Formal prose fine | Natural contractions improve listenability |

**Voice system prompt principles:**
1. **No markdown** — TTS will literally say "asterisk asterisk bold asterisk asterisk"
2. **Contractions** — "don't", "it's", "you're" sound natural spoken
3. **Short turns** — 2–4 sentences unless user explicitly asks for detail
4. **Self-referential** — never say "as shown above" or "see below"
5. **Confirm actions** — "I've set a timer for 5 minutes" rather than silent execution


In [ ]:
# ── Claude voice brain ───────────────────────────────────────────────────────
VOICE_SYSTEM = """You are a helpful voice assistant.

Rules for voice responses:
- Write as you would SPEAK, not type. Use natural contractions (don't, it's, you're).
- No markdown: no **bold**, no - bullet points, no ### headers, no code blocks.
- Keep answers concise: 2-4 sentences unless the user asks for more detail.
- Be conversational: "Great question!" or "Sure thing." are fine.
- For lists, say them naturally: "There are three main options. First... Second... Third..."
- Never say "as shown above" or "see below" — this is voice only.
"""


@dataclass
class ConversationTurn:
    role: str           # "user" or "assistant"
    text: str           # spoken / transcribed text
    latency_s: float = 0.0
    cost_usd: float = 0.0


def voice_think(
    utterance: str,
    history: list,              # list[ConversationTurn]
    model: str = "claude-haiku-4-5",
) -> tuple:                     # (response_text, latency_s, cost_usd)
    """Turn a transcribed utterance into a speakable response."""
    messages = [
        {"role": t.role, "content": t.text}
        for t in history[-6:]   # last 3 exchanges — keep context short for latency
    ]
    messages.append({"role": "user", "content": utterance})

    t0 = time.time()
    resp = client.messages.create(
        model=model,
        max_tokens=256,         # voice answers are short
        system=VOICE_SYSTEM,
        messages=messages,
    )
    latency = time.time() - t0

    text = resp.content[0].text.strip()
    # Haiku pricing: $0.80 input / $4.00 output per M tokens
    cost = (resp.usage.input_tokens * 0.80 + resp.usage.output_tokens * 4.00) / 1_000_000

    return text, latency, cost


# ── Test ──────────────────────────────────────────────────────────────────────
utterance = "Tell me about transformer attention in simple terms."
response_text, lat, cost = voice_think(utterance, history=[])

print(f"User  : {utterance}")
print(f"Claude: {response_text}")
print(f"\nLLM latency: {lat:.3f} s | cost: ${cost:.6f}")


## Part 3 — TTS: Text to Speech

**TTS options** ranked by quality vs accessibility:

| Option | Quality | Latency | Cost | Best for |
|--------|---------|---------|------|----------|
| **gTTS** (Google) | Medium | ~300 ms | Free | Prototyping |
| **pyttsx3** | Low | < 50 ms | Free, offline | Offline / privacy |
| **OpenAI TTS** | High | ~400 ms | $15 / M chars | Production |
| **ElevenLabs** | Highest | ~350 ms | ~$0.30 / 1 K chars | Premium voice |
| **Coqui TTS** | High | ~200 ms | Free, local | Self-hosted |

**Streaming TTS** is the most important optimisation for perceived latency:

```
Non-streaming:  [LLM full response 1.5 s][TTS synthesis 0.5 s][Play ───────]
                ←────────────────────────────────────── 2.0 s of silence

Sentence-chunked streaming:
                [LLM first sentence 0.4 s][TTS chunk 0.2 s][Play chunk 1][chunk 2…]
                ←─────────────────────── 0.6 s of silence  ✓
```

Part 5 of this lesson implements sentence-chunked streaming. Here we start with the simpler batch approach to understand the building blocks.


In [ ]:
# ── TTS: Text to Audio ───────────────────────────────────────────────────────
from gtts import gTTS


def text_to_speech(
    text: str,
    lang: str = "en",
    slow: bool = False,
    output_path: Optional[str] = None,
) -> tuple:                         # (audio_path, synthesis_latency_s)
    """Convert text to a speech MP3 file."""
    if output_path is None:
        output_path = tempfile.mktemp(suffix=".mp3")

    t0 = time.time()
    gTTS(text=text, lang=lang, slow=slow).save(output_path)
    latency = time.time() - t0

    return output_path, latency


def play_audio(audio_path: str):
    """Embed audio player inline in Colab / Jupyter."""
    display(Audio(audio_path, autoplay=False))


# ── Test ──────────────────────────────────────────────────────────────────────
audio_path, tts_lat = text_to_speech(response_text)
print(f"TTS latency: {tts_lat:.3f} s")
print(f"Response   : '{response_text[:100]}...'")
print("\n▶ Audio player:")
play_audio(audio_path)


## Part 4 — The `VoiceAgent` Loop

Now we wire ASR → Claude → TTS into a proper `VoiceAgent` class with:
- **Conversation history** — maintains context across turns
- **Per-stage latency tracking** — ASR / LLM / TTS broken out
- **Cumulative cost meter** — follows the L22 cost-engineering pattern
- **Reliability** — try/except with meaningful error messages


In [ ]:
# ── VoiceAgent ───────────────────────────────────────────────────────────────

@dataclass
class TurnMetrics:
    turn_id: str
    transcript: str
    response: str
    asr_latency_s: float
    llm_latency_s: float
    tts_latency_s: float
    total_latency_s: float
    cost_usd: float

    def summary(self) -> str:
        return (
            f"Turn {self.turn_id[:8]} | "
            f"ASR {self.asr_latency_s:.2f}s + "
            f"LLM {self.llm_latency_s:.2f}s + "
            f"TTS {self.tts_latency_s:.2f}s = "
            f"{self.total_latency_s:.2f}s total | "
            f"${self.cost_usd:.5f}"
        )


class VoiceAgent:
    """
    End-to-end voice agent: audio_file_in → transcribe → think → synthesise → audio_file_out

    Usage:
        agent = VoiceAgent()
        response_audio_path, metrics = agent.turn(input_audio_path)
    """

    def __init__(
        self,
        llm_model: str = "claude-haiku-4-5",
        tts_lang: str = "en",
        max_history_turns: int = 6,
    ):
        self.llm_model = llm_model
        self.tts_lang = tts_lang
        self.max_history_turns = max_history_turns
        self.history: list[ConversationTurn] = []
        self.all_metrics: list[TurnMetrics] = []
        self.total_cost: float = 0.0

    def turn(self, audio_path: str, verbose: bool = True) -> tuple:
        """Process one voice turn. Returns (output_audio_path, TurnMetrics)."""
        turn_id = str(uuid.uuid4())
        t_total = time.time()

        # Stage 1 — ASR
        asr = transcribe_audio(audio_path)
        if verbose:
            print(f"[ASR] '{asr['text']}' ({asr['asr_latency_s']} s)")

        # Stage 2 — LLM
        resp_text, llm_lat, cost = voice_think(
            asr["text"], self.history, model=self.llm_model
        )
        self.total_cost += cost
        if verbose:
            print(f"[LLM] '{resp_text[:70]}...' ({llm_lat:.2f} s)")

        # Stage 3 — TTS
        out_path = f"/tmp/voice_resp_{turn_id[:8]}.mp3"
        _, tts_lat = text_to_speech(resp_text, output_path=out_path)
        if verbose:
            print(f"[TTS] Synthesised ({tts_lat:.2f} s)")

        # Update history (bounded)
        self.history += [
            ConversationTurn("user",      asr["text"], asr["asr_latency_s"]),
            ConversationTurn("assistant", resp_text,   llm_lat, cost),
        ]
        self.history = self.history[-(self.max_history_turns * 2):]

        total = time.time() - t_total
        m = TurnMetrics(
            turn_id=turn_id,
            transcript=asr["text"],
            response=resp_text,
            asr_latency_s=asr["asr_latency_s"],
            llm_latency_s=llm_lat,
            tts_latency_s=tts_lat,
            total_latency_s=total,
            cost_usd=cost,
        )
        self.all_metrics.append(m)
        return out_path, m

    def latency_report(self) -> str:
        if not self.all_metrics:
            return "No turns yet."
        asr  = [m.asr_latency_s  for m in self.all_metrics]
        llm  = [m.llm_latency_s  for m in self.all_metrics]
        tts  = [m.tts_latency_s  for m in self.all_metrics]
        tot  = [m.total_latency_s for m in self.all_metrics]
        rows = [
            "\n══ Latency Report ══════════════════════════════",
            f"  Turns: {len(self.all_metrics)}",
            f"  {'Stage':<10} {'Mean':>8} {'P50':>8}",
            f"  {'-'*30}",
            f"  {'ASR':<10} {statistics.mean(asr):>7.2f}s {statistics.median(asr):>7.2f}s",
            f"  {'LLM':<10} {statistics.mean(llm):>7.2f}s {statistics.median(llm):>7.2f}s",
            f"  {'TTS':<10} {statistics.mean(tts):>7.2f}s {statistics.median(tts):>7.2f}s",
            f"  {'Total':<10} {statistics.mean(tot):>7.2f}s {statistics.median(tot):>7.2f}s",
            f"  Total cost: ${self.total_cost:.5f}",
            "═══════════════════════════════════════════════",
        ]
        return "\n".join(rows)


# ── Demo: 3-turn conversation ─────────────────────────────────────────────────
print("=== 3-Turn Voice Agent Demo ===\n")
agent = VoiceAgent()

utterances = [
    "What is attention in transformer models?",
    "How does self-attention differ from cross-attention?",
    "Give me one practical example of where cross-attention is used.",
]

for i, text in enumerate(utterances, 1):
    print(f"--- Turn {i} ---")
    inp = make_test_audio(text, f"/tmp/user_turn_{i}.mp3")
    out, metrics = agent.turn(inp, verbose=True)
    print(f"→ {metrics.summary()}")
    print(f"   Response: {metrics.response}\n")
    # Uncomment to play audio in Colab:
    # play_audio(out)

print(agent.latency_report())


## Part 5 — Streaming Optimization

The naive pipeline has **sequential** stages: ASR → LLM (full generation) → TTS → Play.
Even with fast models this leaves 1.5–3 s of silence before the user hears anything.

**Sentence-chunked streaming** overlaps the stages:

```
Sequential:   [ASR 0.3s][LLM full 0.8s][TTS 0.3s][Play ────────]
              ←───────────────────────────────────── 1.4 s silence

Streaming:    [ASR 0.3s][LLM token stream ───────────────────── ]
                              ├─[TTS chunk 1 0.2s][▶ Play ch.1]
                                        ├─[TTS chunk 2 0.2s][▶ ch.2]
                                                  ├─[▶ ch.3...]
              ←─────────────── 0.7 s to first sound  ✓
```

**Algorithm:**
1. Stream LLM output token-by-token via `client.messages.stream()`
2. Accumulate tokens in a buffer
3. On sentence-boundary detection (`.`, `?`, `!`, `;`) — send the complete sentence to TTS
4. TTS synthesises chunk N while chunk N-1 is playing
5. User hears continuous audio with no perceptible gap


In [ ]:
# ── Streaming voice agent ────────────────────────────────────────────────────
SENTENCE_END = re.compile(r'(?<=[.!?;])\s+')


def stream_voice_response(
    utterance: str,
    history: list,
    model: str = "claude-haiku-4-5",
    verbose: bool = True,
) -> tuple:         # (audio_chunk_paths, total_latency_s, cost_usd)
    """
    Stream LLM output → sentence-chunked TTS for minimum perceived latency.
    Prints a timestamp for each chunk so you can see the streaming win.
    """
    t0 = time.time()
    messages = [{"role": t.role, "content": t.text} for t in history[-6:]]
    messages.append({"role": "user", "content": utterance})

    buffer = ""
    audio_chunks: list[str] = []

    with client.messages.stream(
        model=model,
        max_tokens=256,
        system=VOICE_SYSTEM,
        messages=messages,
    ) as stream:
        for token in stream.text_stream:
            buffer += token

            # Flush complete sentences to TTS immediately
            parts = SENTENCE_END.split(buffer)
            if len(parts) > 1:
                for sentence in parts[:-1]:
                    sentence = sentence.strip()
                    if len(sentence) > 8:           # skip very short fragments
                        chunk_path = f"/tmp/stream_chunk_{len(audio_chunks)}.mp3"
                        text_to_speech(sentence, output_path=chunk_path)
                        audio_chunks.append(chunk_path)
                        if verbose:
                            elapsed = time.time() - t0
                            print(f"  [chunk {len(audio_chunks)}] ready at {elapsed:.2f}s "
                                  f"— '{sentence[:55]}'")
                buffer = parts[-1]  # keep the incomplete sentence

    # Flush remaining buffer
    if buffer.strip() and len(buffer.strip()) > 5:
        chunk_path = f"/tmp/stream_chunk_{len(audio_chunks)}.mp3"
        text_to_speech(buffer.strip(), output_path=chunk_path)
        audio_chunks.append(chunk_path)

    usage = stream.get_final_message().usage
    cost  = (usage.input_tokens * 0.80 + usage.output_tokens * 4.00) / 1_000_000
    total = time.time() - t0
    return audio_chunks, total, cost


# ── Compare sequential vs streaming ──────────────────────────────────────────
print("=== Sequential vs Streaming ===\n")
Q = "Explain the difference between supervised and unsupervised learning in 3 sentences."

# Sequential
t0 = time.time()
resp, llm_lat, _ = voice_think(Q, [])
_, tts_lat        = text_to_speech(resp)
seq_total         = time.time() - t0
print(f"Sequential: {seq_total:.2f}s total")
print(f"  Time-to-first-sound: LLM {llm_lat:.2f}s + TTS {tts_lat:.2f}s = {llm_lat+tts_lat:.2f}s\n")

# Streaming
print("Streaming:")
chunks, stream_total, _ = stream_voice_response(Q, [], verbose=True)
print(f"\nStreaming: {stream_total:.2f}s total | {len(chunks)} audio chunks")
print(f"  Time-to-first-sound: see chunk 1 timestamp above")
print(f"\nConclusion: even if totals are similar, streaming cuts perceived latency ~50%")


## Part 6 — Latency Budget & Profiling

**Design principle:** Start with a latency target, then work backward to model selection.

```
Target: < 1,500 ms time-to-first-sound

Budget:
  ASR   (Whisper base, 5 s audio, T4 GPU)  ≈  300 ms
  Network RTT to LLM API                   ≈   50 ms
  LLM TTFT (Haiku)                         ≈  400 ms
  TTS first chunk synthesis                ≈  200 ms
  Playback buffer fill                     ≈   50 ms
  ─────────────────────────────────────────────────
  Total                                    ≈ 1000 ms  ✓
```

**Optimisation levers when over budget:**

| Lever | Savings | Tradeoff |
|-------|---------|----------|
| Whisper `base` → `tiny` | −100 ms | ~5 % worse WER |
| Streaming ASR (real-time) | −200 ms | More complex code |
| Prompt caching (L22) | −80 ms | Cache miss on first call |
| Haiku over Sonnet | −150 ms | Capability drop |
| Sentence-chunked streaming TTS | −300 ms perceived | More complex code |
| Reduce `max_history_turns` | −50 ms | Context loss |


In [ ]:
# ── LatencyBudget profiler ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


@dataclass
class LatencyBudget:
    target_ms: float
    asr_ms: float
    llm_ttft_ms: float          # time-to-first-token (not total generation)
    tts_first_chunk_ms: float
    playback_ms: float = 50.0

    @property
    def total_ms(self) -> float:
        return self.asr_ms + self.llm_ttft_ms + self.tts_first_chunk_ms + self.playback_ms

    @property
    def headroom_ms(self) -> float:
        return self.target_ms - self.total_ms

    def bar_chart(self, title: str = "Latency Budget"):
        stages  = ["ASR", "LLM TTFT", "TTS first", "Playback"]
        values  = [self.asr_ms, self.llm_ttft_ms, self.tts_first_chunk_ms, self.playback_ms]
        colors  = ["#4C9BE8", "#F5A623", "#7ED321", "#9B59B6"]
        cumul   = np.cumsum([0] + values[:-1])

        fig, ax = plt.subplots(figsize=(9, 2.5))
        for i, (s, v, c, x) in enumerate(zip(stages, values, colors, cumul)):
            ax.barh(0, v, left=x, color=c, edgecolor="white", height=0.5)
            if v > 30:
                ax.text(x + v / 2, 0, f"{v:.0f}ms", ha="center", va="center",
                        fontsize=9, color="white", fontweight="bold")

        ax.axvline(self.target_ms, color="red", linewidth=2, linestyle="--", label=f"Target {self.target_ms:.0f}ms")
        ax.axvline(self.total_ms,  color="black", linewidth=1.5, linestyle="-",  label=f"Total  {self.total_ms:.0f}ms")
        ax.set_xlim(0, max(self.target_ms * 1.2, self.total_ms * 1.1))
        ax.set_yticks([])
        ax.set_xlabel("Milliseconds")
        status = "✓ Within budget" if self.headroom_ms >= 0 else f"✗ Over by {-self.headroom_ms:.0f}ms"
        ax.set_title(f"{title}  —  {status}  (headroom {self.headroom_ms:+.0f}ms)")
        patches = [mpatches.Patch(color=c, label=s) for s, c in zip(stages, colors)]
        ax.legend(handles=patches + [
            plt.Line2D([0],[0], color="red",   linestyle="--", label="Target"),
            plt.Line2D([0],[0], color="black", linestyle="-",  label="Total"),
        ], loc="upper right", fontsize=8)
        plt.tight_layout()
        plt.show()


# Use measured latencies from the demo
if agent.all_metrics:
    avg_asr = statistics.mean([m.asr_latency_s * 1000 for m in agent.all_metrics])
    avg_llm = statistics.mean([m.llm_latency_s * 1000 for m in agent.all_metrics])
    avg_tts = statistics.mean([m.tts_latency_s * 1000 for m in agent.all_metrics])
else:
    avg_asr, avg_llm, avg_tts = 400, 700, 350     # fallback estimates

# Sequential budget
seq_budget = LatencyBudget(target_ms=1500, asr_ms=avg_asr,
                            llm_ttft_ms=avg_llm, tts_first_chunk_ms=avg_tts)
seq_budget.bar_chart("Sequential Pipeline (full LLM + full TTS)")

# Streaming budget (TTFT only + first sentence TTS)
stream_budget = LatencyBudget(target_ms=1500, asr_ms=avg_asr,
                               llm_ttft_ms=min(avg_llm, 350),
                               tts_first_chunk_ms=min(avg_tts, 200))
stream_budget.bar_chart("Streaming Pipeline (TTFT + first-sentence TTS only)")

print(f"Sequential time-to-first-sound: {seq_budget.total_ms:.0f} ms")
print(f"Streaming  time-to-first-sound: {stream_budget.total_ms:.0f} ms")
print(f"Streaming saves: {seq_budget.total_ms - stream_budget.total_ms:.0f} ms perceived latency")


## Part 7 — Production Patterns

Three patterns separate a "demo that works" from a "product people use":

### 1. Voice Activity Detection (VAD)
Don't start transcribing until the user is actually speaking. **Silero VAD** is the standard choice — 10 ms inference, 1.7 MB model, works in real-time.

```python
# Production snippet (requires: pip install silero-vad)
import torch
vad_model, utils = torch.hub.load('snakers4/silero-vad', 'silero_vad')
get_speech_timestamps, _, read_audio, *_ = utils

def get_speech_segments(audio_path: str) -> list[dict]:
    wav = read_audio(audio_path, sampling_rate=16_000)
    return get_speech_timestamps(wav, vad_model, sampling_rate=16_000)
    # Returns: [{"start": 1600, "end": 12800}, ...]   (sample indices)
```

### 2. Turn Detection
Distinguish "user paused mid-sentence" from "user finished speaking":
- Short pause (< 500 ms after speech end): Continue listening
- Long pause (> 800 ms): Trigger ASR + LLM pipeline
- Interruption detected: User speaks while TTS is playing → barge-in

### 3. Barge-in / Interruption Handling
```python
class InterruptiblePlayer:
    def __init__(self):
        self._stop = threading.Event()

    def play(self, chunks: list[str]):
        self._stop.clear()
        for chunk in chunks:
            if self._stop.is_set():
                break
            # sounddevice.play(chunk); sounddevice.wait()

    def interrupt(self):
        """Call from VAD thread when user starts speaking."""
        self._stop.set()
```

### Hot-mic Loop Prevention
Gate the microphone input while TTS is playing. Otherwise ASR transcribes the assistant's own speech and creates an infinite loop.

```python
class VoicePipeline:
    def turn(self, audio: bytes):
        transcript = asr(audio)
        response = llm(transcript)
        self._mic_gate.close()         # ← stop listening
        play_tts(response)
        self._mic_gate.open()          # ← resume listening
```


In [ ]:
# ── Interruption-aware player (Colab simulation) ─────────────────────────────
# In production: replace sleep with sounddevice.play() + sounddevice.wait()

class InterruptibleTTSPlayer:
    """
    Plays TTS audio chunks sequentially. Can be interrupted mid-playback.
    Colab simulation: uses IPython.display.Audio + short sleep.
    Production: replace _play_chunk() with sounddevice or pygame playback.
    """
    def __init__(self):
        self._stop = threading.Event()
        self.is_playing = False

    def play_chunks(self, chunk_paths: list[str], verbose: bool = True):
        self._stop.clear()
        self.is_playing = True
        for i, path in enumerate(chunk_paths):
            if self._stop.is_set():
                if verbose:
                    print(f"  [Player] ✗ interrupted after chunk {i}")
                break
            if verbose:
                print(f"  [Player] ▶ chunk {i+1}/{len(chunk_paths)}: {Path(path).name}")
            display(Audio(path, autoplay=False))
            time.sleep(0.15)            # simulate playback time (not real duration)
        self.is_playing = False

    def interrupt(self):
        """Stop playback — call from VAD thread when user starts speaking."""
        self._stop.set()


# ── Demo: user barges in after chunk 1 ───────────────────────────────────────
available_chunks = [p for p in
    [f"/tmp/stream_chunk_{i}.mp3" for i in range(4)]
    if Path(p).exists()
]

if len(available_chunks) >= 2:
    player = InterruptibleTTSPlayer()
    print("=== Barge-in Demo ===\n")
    print("Starting TTS playback...")

    play_thread = threading.Thread(
        target=player.play_chunks,
        args=(available_chunks,),
        daemon=True,
    )
    play_thread.start()

    time.sleep(0.5)     # simulate user speaking after 0.5 s
    print("\n[VAD] Speech detected → interrupting TTS...")
    player.interrupt()
    play_thread.join()
    print("[System] Playback stopped. Listening for new utterance.")
else:
    print("(Run the streaming cell first to generate audio chunks.)")


## 10 Pitfalls in Voice Agent Development

| # | Pitfall | Symptom | Fix |
|---|---------|---------|-----|
| 1 | **No VAD** | ASR transcribes silence → hallucinated words | Use Silero VAD before feeding audio |
| 2 | **Markdown in LLM output** | TTS says "asterisk asterisk bold asterisk asterisk" | Explicit no-markdown system prompt |
| 3 | **No latency budget** | Ship a 4 s agent; users abandon after 2 turns | Measure TTFT at design time |
| 4 | **Growing history** | Latency grows with each turn | Rolling 6-turn `max_history_turns` cap |
| 5 | **Sequential pipeline** | 2 s silence before first word | Sentence-chunked streaming TTS |
| 6 | **No barge-in** | User can't interrupt a long TTS playback | `InterruptiblePlayer.interrupt()` |
| 7 | **Wrong Whisper model** | `large-v3` on CPU = 45× real-time = unusable | `base` on T4; `tiny` on CPU |
| 8 | **Hot-mic loop** | Agent transcribes its own TTS output | Gate mic while TTS is playing |
| 9 | **`fp16=True` on CPU** | `AssertionError` from Whisper | Always `fp16=False` without GPU |
| 10 | **Cost blindness** | Voice seems cheap per turn; expensive at scale | Track `cost_usd` per turn; Haiku for voice |


## 📚 Homework

1. **Swap ASR backends:** Replace local Whisper with the **OpenAI Whisper API** (`openai.audio.transcriptions.create`). Compare latency and output on the same test clip. Which is faster? Which handles accents better?

2. **Add tools to the voice brain:** Give `VoiceAgent` a `get_time()` tool and a `get_weather(city: str)` tool (mock data is fine). Test: "What time is it?" and "What's the weather in Tokyo?"

3. **Language auto-detection:** Whisper returns `result["language"]`. Modify `voice_think` to respond in the same language the user spoke. Test with a French or Spanish utterance.

4. **Latency stacked bar chart:** Run 10 turns and plot ASR / LLM / TTS as a stacked horizontal bar chart with matplotlib. Annotate which stage is the bottleneck.

5. **FastAPI `/voice` endpoint:** Create a FastAPI app with a `POST /voice` route that accepts an audio file upload, processes one `VoiceAgent` turn, and returns the response audio as `audio/mpeg`. This turns `VoiceAgent` into a REST microservice compatible with any front end.

---

## Track 4 Roadmap

| Lesson | Topic | What you'll build |
|--------|-------|-------------------|
| **L42** ← *today* | **Voice Pipelines** | ASR + Claude + TTS, streaming, latency profiler |
| L43 | **Image Generation as Agent Tools** | Agent that generates images via `image_gen` tool |
| L44 | **Document AI** | PDF extraction + OCR + visual Q&A pipeline |
| L45 | **Track 4 Capstone** | Multimodal agent: voice query → search + vision → spoken summary |

**Building toward open-source:** By L45, the AutoResearcher will accept voice input, read PDFs and images, generate diagrams, and speak its summary back — a genuinely multimodal AI agent.

---
*Lesson 42 complete. See you next run for L43 — Image Generation as Agent Tools.*
